Modified Cosine Similarity


In [1]:
from pathlib import Path
import pickle
import time
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from matchms.similarity import ModifiedCosine

DATA = Path.cwd().parent / "data" if Path.cwd().name == "scripts" else Path.cwd() / "data"
assert DATA.exists(), f"Data directory not found: {DATA}. Run from the repository root or scripts directory."
print("Data directory:", DATA.resolve())

Data directory: C:\Users\aidan\OneDrive\Desktop\bioinformatics_project\data


In [2]:
with (DATA / "pesticides_clean_spectra.pkl").open("rb") as f:
    pest_spectra = pickle.load(f)
with (DATA / "pharmaceuticals_clean_spectra.pkl").open("rb") as f:
    pharma_spectra = pickle.load(f)

pest_truth = np.load(DATA / "pesticides_tanimoto_matrix.npy")
pharma_truth = np.load(DATA / "pharmaceuticals_tanimoto_matrix.npy")
assert len(pest_spectra) == pest_truth.shape[0] == pest_truth.shape[1]
assert len(pharma_spectra) == pharma_truth.shape[0] == pharma_truth.shape[1]
print("Pesticides:", len(pest_spectra), "Pharmaceuticals:", len(pharma_spectra))

Pesticides: 653 Pharmaceuticals: 377


Calculate Modified Cosine Scores


In [3]:
similarity = ModifiedCosine(tolerance=0.1, mz_power=0.0, intensity_power=0.5)

def compute_similarity_matrix(spectra, name):
    n = len(spectra)
    matrix = np.eye(n, dtype=np.float32)
    start = time.perf_counter()
    for i in range(n):
        if i % 50 == 0:
            print(f"{name}: {i}/{n} spectra")
        for j in range(i + 1, n):
            result = similarity.pair(spectra[i], spectra[j])
            # matchms versions may return a dict or a structured NumPy scalar.
            score = float(result["score"])
            matrix[i, j] = matrix[j, i] = score
    elapsed = time.perf_counter() - start
    assert np.allclose(matrix, matrix.T)
    assert np.allclose(np.diag(matrix), 1.0)
    print(f"{name}: {n*(n-1)//2:,} unique pairs, {elapsed:.2f} seconds")
    return matrix, elapsed

pest_matrix, pest_seconds = compute_similarity_matrix(pest_spectra, "Pesticides")
pharma_matrix, pharma_seconds = compute_similarity_matrix(pharma_spectra, "Pharmaceuticals")
np.save(DATA / "pest_modified_cosine_matrix.npy", pest_matrix)
np.save(DATA / "pharma_modified_cosine_matrix.npy", pharma_matrix)

Pesticides: 0/653 spectra
Pesticides: 50/653 spectra
Pesticides: 100/653 spectra
Pesticides: 150/653 spectra
Pesticides: 200/653 spectra
Pesticides: 250/653 spectra
Pesticides: 300/653 spectra
Pesticides: 350/653 spectra
Pesticides: 400/653 spectra
Pesticides: 450/653 spectra
Pesticides: 500/653 spectra
Pesticides: 550/653 spectra
Pesticides: 600/653 spectra
Pesticides: 650/653 spectra
Pesticides: 212,878 unique pairs, 8.96 seconds
Pharmaceuticals: 0/377 spectra
Pharmaceuticals: 50/377 spectra
Pharmaceuticals: 100/377 spectra
Pharmaceuticals: 150/377 spectra
Pharmaceuticals: 200/377 spectra
Pharmaceuticals: 250/377 spectra
Pharmaceuticals: 300/377 spectra
Pharmaceuticals: 350/377 spectra
Pharmaceuticals: 70,876 unique pairs, 8.43 seconds


Correlation with Tanimoto Similarity


In [4]:
def evaluate_alignment(truth, predicted, name):
    assert truth.shape == predicted.shape
    indices = np.triu_indices_from(truth, k=1)
    actual, scores = truth[indices], predicted[indices]
    pearson = pearsonr(actual, scores).statistic
    spearman = spearmanr(actual, scores).statistic
    print(f"{name}: {len(actual):,} pairs | Pearson r={pearson:.4f} | Spearman rho={spearman:.4f}")
    return {"dataset": name, "pairs": len(actual), "pearson": pearson, "spearman": spearman}

results = [evaluate_alignment(pest_truth, pest_matrix, "Pesticides"),
           evaluate_alignment(pharma_truth, pharma_matrix, "Pharmaceuticals")]
pd.DataFrame(results)

Pesticides: 212,878 pairs | Pearson r=0.1986 | Spearman rho=0.0035
Pharmaceuticals: 70,876 pairs | Pearson r=0.1272 | Spearman rho=0.1073


,dataset,pairs,pearson,spearman
0,Pesticides,212878,0.198558,0.003464
1,Pharmaceuticals,70876,0.127152,0.107302


Scores by Structural Similarity Bin


In [5]:
def stratify_scores(truth, predicted, name):
    indices = np.triu_indices_from(truth, k=1)
    df = pd.DataFrame({"tanimoto": truth[indices], "modified_cosine": predicted[indices]})
    bins = [0, .2, .4, .6, .8, 1.0]
    df["bin"] = pd.cut(df["tanimoto"], bins=bins, include_lowest=True)
    summary = df.groupby("bin", observed=False)["modified_cosine"].agg(["count", "mean", "median", "std"])
    print(name)
    display(summary)
    return summary

pest_bins = stratify_scores(pest_truth, pest_matrix, "Pesticides")
pharma_bins = stratify_scores(pharma_truth, pharma_matrix, "Pharmaceuticals")

Pesticides


,count,mean,median,std
bin,,,,
"(-0.001, 0.2]",193301,0.198650,0.150923,0.168029
"(0.2, 0.4]",16193,0.241741,0.178464,0.210429
"(0.4, 0.6]",1636,0.411445,0.350364,0.276755
"(0.6, 0.8]",520,0.547646,0.540146,0.291913
"(0.8, 1.0]",1228,0.605303,0.739154,0.328514


Pharmaceuticals


,count,mean,median,std
bin,,,,
"(-0.001, 0.2]",68744,0.395207,0.393623,0.174030
"(0.2, 0.4]",1889,0.457518,0.468908,0.183364
"(0.4, 0.6]",130,0.598869,0.636255,0.194259
"(0.6, 0.8]",48,0.646401,0.672232,0.153244
"(0.8, 1.0]",65,0.576917,0.592544,0.174475


Compare with Spec2Vec


In [6]:
for name, truth, matrix, spec_file in [
    ("Pesticides", pest_truth, pest_matrix, "pest_spec2vec_matrix.npy"),
    ("Pharmaceuticals", pharma_truth, pharma_matrix, "pharma_spec2vec_matrix.npy")]:
    path = DATA / spec_file
    if path.exists():
        spec = np.load(path)
        print(f"\n{name}")
        evaluate_alignment(truth, matrix, "Modified cosine")
        evaluate_alignment(truth, spec, "Spec2Vec")


Pesticides
Modified cosine: 212,878 pairs | Pearson r=0.1986 | Spearman rho=0.0035
Spec2Vec: 212,878 pairs | Pearson r=0.4070 | Spearman rho=0.1107

Pharmaceuticals
Modified cosine: 70,876 pairs | Pearson r=0.1272 | Spearman rho=0.1073
Spec2Vec: 70,876 pairs | Pearson r=0.2262 | Spearman rho=0.1776


Different-Molecule Comparison


In [7]:
from rdkit import Chem

def load_metadata(filename, n):
    metadata = pd.read_csv(DATA / filename).sort_values("matrix_index").reset_index(drop=True)
    assert len(metadata) == n
    assert np.array_equal(metadata["matrix_index"].to_numpy(), np.arange(n))
    return metadata

def canonical_smiles(smiles):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(str(smiles))
    return Chem.MolToSmiles(mol) if mol is not None else None

pest_meta = load_metadata("pesticides_metadata.csv", len(pest_spectra))
pharma_meta = load_metadata("pharmaceuticals_metadata.csv", len(pharma_spectra))
pest_smiles = np.array([canonical_smiles(s) for s in pest_meta["smiles"]], dtype=object)
pharma_smiles = np.array([canonical_smiles(s) for s in pharma_meta["smiles"]], dtype=object)

def distinct_pair_metrics(truth, matrix, smiles, name):
    i, j = np.triu_indices(len(smiles), k=1)
    valid = np.array([(smiles[a] is not None and smiles[b] is not None) for a, b in zip(i, j)])
    same = valid & (smiles[i] == smiles[j])
    different = valid & ~same
    print(f"{name}: {same.sum():,} same-molecule pairs; {different.sum():,} different-molecule pairs")
    for label, mask in [("Different molecules", different), ("Same molecule", same)]:
        if mask.sum() < 3:
            continue
        print(f"  {label}: median score={np.median(matrix[i[mask], j[mask]]):.4f}")
        if label == "Different molecules":
            evaluate_alignment_values(truth[i[mask], j[mask]], matrix[i[mask], j[mask]], label)
    return different

def evaluate_alignment_values(actual, scores, name):
    if len(actual) < 3 or np.std(actual) == 0 or np.std(scores) == 0:
        print(f"{name}: insufficient variation for correlation")
        return
    print(f"{name}: {len(actual):,} pairs | Pearson r={pearsonr(actual, scores).statistic:.4f} | Spearman rho={spearmanr(actual, scores).statistic:.4f}")

for name, truth, modcos, smiles, spec_file in [
    ("Pesticides", pest_truth, pest_matrix, pest_smiles, "pest_spec2vec_matrix.npy"),
    ("Pharmaceuticals", pharma_truth, pharma_matrix, pharma_smiles, "pharma_spec2vec_matrix.npy")]:
    print(f"\n{name} — modified cosine")
    distinct_pair_metrics(truth, modcos, smiles, name)
    path = DATA / spec_file
    if path.exists():
        i, j = np.triu_indices(len(smiles), k=1)
        mask = np.array([smiles[a] is not None and smiles[b] is not None and smiles[a] != smiles[b] for a, b in zip(i, j)])
        print(f"{name} — Spec2Vec, different molecules")
        evaluate_alignment_values(truth[i[mask], j[mask]], np.load(path)[i[mask], j[mask]], "Spec2Vec")



Pesticides — modified cosine
Pesticides: 1,129 same-molecule pairs; 211,749 different-molecule pairs
  Different molecules: median score=0.1541
Different molecules: 211,749 pairs | Pearson r=0.1221 | Spearman rho=-0.0072
  Same molecule: median score=0.7653
Pesticides — Spec2Vec, different molecules
Spec2Vec: 211,749 pairs | Pearson r=0.2693 | Spearman rho=0.1006

Pharmaceuticals — modified cosine
Pharmaceuticals: 55 same-molecule pairs; 70,821 different-molecule pairs
  Different molecules: median score=0.3963
Different molecules: 70,821 pairs | Pearson r=0.1298 | Spearman rho=0.1063
  Same molecule: median score=0.5920
Pharmaceuticals — Spec2Vec, different molecules
Spec2Vec: 70,821 pairs | Pearson r=0.2199 | Spearman rho=0.1760


Top-1 Neighbor Comparison


In [8]:
def retrieval_metrics(truth, scores, smiles, name):
    valid = np.array([s is not None for s in smiles])
    allowed = valid[:, None] & valid[None, :] & (smiles[:, None] != smiles[None, :])
    rows = np.flatnonzero(allowed.any(axis=1))
    if not len(rows):
        print(f"{name}: no eligible queries")
        return
    masked_scores = np.where(allowed, scores, -np.inf)
    neighbors = masked_scores[rows].argmax(axis=1)
    selected = truth[rows, neighbors]
    oracle = np.where(allowed, truth, -np.inf)[rows].max(axis=1)
    print(f"{name}: {len(rows)} queries | top-1 mean Tanimoto={selected.mean():.4f} | best available mean={oracle.mean():.4f}")

for name, truth, matrix, smiles, spec_file in [
    ("Pesticides", pest_truth, pest_matrix, pest_smiles, "pest_spec2vec_matrix.npy"),
    ("Pharmaceuticals", pharma_truth, pharma_matrix, pharma_smiles, "pharma_spec2vec_matrix.npy")]:
    retrieval_metrics(truth, matrix, smiles, f"{name} modified cosine")
    if (DATA / spec_file).exists():
        retrieval_metrics(truth, np.load(DATA / spec_file), smiles, f"{name} Spec2Vec")


Pesticides modified cosine: 653 queries | top-1 mean Tanimoto=0.2920 | best available mean=0.4631
Pesticides Spec2Vec: 653 queries | top-1 mean Tanimoto=0.2914 | best available mean=0.4631
Pharmaceuticals modified cosine: 377 queries | top-1 mean Tanimoto=0.2482 | best available mean=0.4208
Pharmaceuticals Spec2Vec: 377 queries | top-1 mean Tanimoto=0.2482 | best available mean=0.4208


Instrument Comparison


In [9]:
instrument = pest_meta["instrument_type"].fillna("").str.lower()
family = np.where(instrument.str.contains("orbitrap|q-exactive"), "Orbitrap",
                  np.where(instrument.str.contains("qtof|q-tof|qtof|tof"), "qTOF", "Other"))
i, j = np.triu_indices(len(pest_smiles), k=1)
same = np.array([pest_smiles[a] is not None and pest_smiles[b] is not None and pest_smiles[a] == pest_smiles[b] for a, b in zip(i, j)])
for label, mask in [
    ("Within Orbitrap", same & (family[i] == "Orbitrap") & (family[j] == "Orbitrap")),
    ("Within qTOF", same & (family[i] == "qTOF") & (family[j] == "qTOF")),
    ("Orbitrap vs qTOF", same & (((family[i] == "Orbitrap") & (family[j] == "qTOF")) | ((family[i] == "qTOF") & (family[j] == "Orbitrap"))))]:
    if mask.any():
        print(f"{label}: {mask.sum():,} pairs | mean={pest_matrix[i[mask], j[mask]].mean():.4f} | median={np.median(pest_matrix[i[mask], j[mask]]):.4f}")
    else:
        print(f"{label}: no eligible pairs")


Within Orbitrap: 437 pairs | mean=0.6510 | median=0.9066
Within qTOF: 77 pairs | mean=0.3666 | median=0.2901
Orbitrap vs qTOF: 615 pairs | mean=0.6149 | median=0.7802


Runtime


In [10]:
pd.DataFrame([
    {"dataset": "Pesticides", "spectra": len(pest_spectra), "pairs": len(pest_spectra)*(len(pest_spectra)-1)//2, "seconds": pest_seconds},
    {"dataset": "Pharmaceuticals", "spectra": len(pharma_spectra), "pairs": len(pharma_spectra)*(len(pharma_spectra)-1)//2, "seconds": pharma_seconds},
])

,dataset,spectra,pairs,seconds
0,Pesticides,653,212878,8.959951
1,Pharmaceuticals,377,70876,8.429369
